# Homework 2: SPY on a 1-minute chart

The assignment has two parts: clean and analyze the data, then build a simple correlation trading rule. I’m using SPY and keeping the main test at 1 minute. There’s enough trading in SPY to try that without starting from 5-minute bars.

I want to see whether an indicator has any relationship with the return that comes **after** it. A smooth line on a chart by itself doesn’t answer that.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
ROOT = Path.cwd()
REBUILD = False
BAR_MINUTES = 1
EMA_MINUTES = 20
RSI_MINUTES = 14
ATR_MINUTES = 14
CORR_MINUTES = 60
CORR_CUTOFF = 0.10
COST_BPS = 0.5

plt.rcParams.update({"figure.dpi": 130, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.18})

## a) Start with the trades

The downloads are huge, so I read one in chunks. Before making bars, I keep SPY trades with a valid time, positive price and size, and no correction/cancel flag. I also remove exact repeated records and prints that are late, out of sequence, or based on an average/reference price. Odd lots stay in: a small trade isn’t automatically a bad trade.

The session here is 9:30 to just before 16:00, New York time. The source timestamps are already Eastern time; they are not UTC. I check the file’s ordering rather than assuming the first row is always the earliest trade.

The chunk-reading code is in `spy_data.py` so the notebook doesn’t turn into a wall of file-handling code. The filter and trading calculations are below.

### Why 1 minute?

Five minutes is a choice, not a requirement. A full session gives 390 one-minute bars versus 78 five-minute bars. I’d rather start at 1 minute and check what happens when I aggregate it later.

The trade-off is noise: a 1-minute move is small, and the spread can eat a lot of it. More bars also doesn’t mean more independent evidence. We still only have a few trading days.

Each bar is labeled by its **start**. The 9:30 bar covers 9:30:00–9:30:59.999…; its close is only known at 9:31. OHLC uses the actual trade order, volume is summed, and VWAP is total price × size divided by total size.

In [ ]:
from spy_data import load_bars

bars, audit = load_bars(ROOT, rebuild=REBUILD)
print(f"{audit['raw_rows']:,} trade records → {audit['kept_rows']:,} kept")
print(f"{len(bars):,} minute bars across {audit['sessions']} sessions")
print(f"{bars.index[0].date()} to {bars.index[-1].date()}")
print("Identical downloads:", audit["duplicate_download"])
print("Times are New York time; no missing minute is filled in.")
display(bars.head().round(4))

audit_table = pd.DataFrame({"records removed": audit["removed"]})
audit_table.loc["kept", "records removed"] = audit["kept_rows"]
display(audit_table.astype(int))